In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

In [ ]:
# Load the data
df = pd.read_csv("train.csv")

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows where close is missing
df = df[df["close"].notna()]

# Separate target variable
y = df["close"]

# Separate features
X = df.drop(columns=["close"])

In [ ]:
for column in X.select_dtypes(include=["number"]).columns:
    plt.figure(figsize=(8, 5))
    plt.scatter(X[column], y, s=3, alpha=0.3)
    plt.xlabel(column)
    plt.ylabel("Close")
    plt.title(f"{column} vs Close")
    plt.show()

In [ ]:
# Remove unwanted columns (the graphs made no sense)
X = X.drop(columns=["index", "volume", "quant_index"])

# Remove non numeric columns
X = X.select_dtypes(include=["number"])

# Impute missing values in X
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Convert X and y to NumPy
X_np = X.values.astype(float)
y_np = y.values.astype(float)

# Scaling X
X_mean = np.mean(X_np, axis=0)
X_std = np.std(X_np, axis=0)

X_scaled = (X_np - X_mean) / X_std

In [ ]:
def gradient_descent(X, y, learning_rate=0.01,
                     tolerance=1e-6, max_iterations=100000):

    m = len(y)
    n = X.shape[1]

    w = np.zeros(n)
    b = 0

    previous_cost = float('inf')

    for i in range(max_iterations):

        y_pred = X.dot(w) + b
        error = y_pred - y

        cost = (1 / (2 * m)) * np.sum(error ** 2)

        dw = (1 / m) * X.T.dot(error)
        db = (1 / m) * np.sum(error)

        w = w - learning_rate * dw
        b = b - learning_rate * db

        if abs(previous_cost - cost) < tolerance:
            break

        previous_cost = cost

    return w, b, i + 1

In [ ]:
w, b, iterations = gradient_descent(X_scaled, y_np)

print("Weights:", w)
print("Bias:", b)
print("Iterations:", iterations)

In [ ]:
# Load test data
test_df = pd.read_csv("test.csv")

# Remove duplicate rows
test_df = test_df.drop_duplicates()

# Remove rows where close is missing
test_df = test_df[test_df["close"].notna()]

# Separate target
y_test = test_df["close"]

# Separate features
X_test = test_df.drop(columns=["close"])

# Use the SAME columns, in the SAME order, as the training data
X_test = X_test[X.columns]

# Handle missing values (using the median from TRAINING data)
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

# Convert to NumPy
X_test_np = X_test.values.astype(float)
y_test_np = y_test.values.astype(float)

# Scale test data using the TRAINING mean and std
X_test_scaled = (X_test_np - X_mean) / X_std

In [ ]:
# Make predictions
y_pred = X_test_scaled.dot(w) + b

# R² score
ss_total = np.sum((y_test_np - np.mean(y_test_np)) ** 2)
ss_residual = np.sum((y_test_np - y_pred) ** 2)

r2 = 1 - (ss_residual / ss_total)

print("R² Score:", r2)
print("Model Accuracy:", r2 * 100, "%")